In [1]:
import math
import random
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# -----------------------------
# 1) Datos: texto -> ids
# -----------------------------
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def build_vocab(text: str):
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)

In [2]:
# -----------------------------
# 2) Dataset de ventanas
# -----------------------------
class CharWindowDataset(Dataset):
    def __init__(self, data_ids: torch.Tensor, seq_len: int):
        self.data = data_ids
        self.seq_len = seq_len

    def __len__(self):
        # -1 porque necesitamos un target corrido
        return max(0, len(self.data) - self.seq_len - 1)

    def __getitem__(self, idx: int):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y

In [3]:
# -----------------------------
# 3) Modelo baseline: Embedding + GRU + Linear
# -----------------------------
class CharGRULM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 128, hidden_dim: int = 256, num_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h0=None):
        # x: (B, T)
        emb = self.embed(x)              # (B, T, E)
        out, hN = self.gru(emb, h0)      # out: (B, T, H)
        logits = self.fc(out)           # (B, T, V)
        return logits, hN

In [4]:
# -----------------------------
# 4) Entrenamiento
# -----------------------------
@dataclass
class TrainConfig:
    seq_len: int = 128
    batch_size: int = 64
    emb_dim: int = 128
    hidden_dim: int = 256
    num_layers: int = 2
    lr: float = 3e-4
    epochs: int = 5
    device: str = "cpu"

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0

    for x, y in loader:
        x = x.to(device)  # (B,T)
        y = y.to(device)  # (B,T)

        optimizer.zero_grad()
        logits, _ = model(x)  # (B,T,V)

        # CrossEntropyLoss espera (N, C) y target (N)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()

        # Grad clipping ayuda estabilidad
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(1, n_batches)

@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, _ = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)

In [5]:
# -----------------------------
# 5) Generación (sampling con temperatura)
# -----------------------------
@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, max_new_tokens: int = 400, temperature: float = 1.0, device="cpu"):
    model.eval()

    # estado inicial
    h = None

    # codificar prompt
    ids = [stoi[ch] for ch in prompt if ch in stoi]
    if not ids:
        # si el prompt no contiene chars del vocab, usar un char cualquiera
        ids = [0]

    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)  # (1,Tprompt)
    logits, h = model(x, h0=h)

    # tomamos el último logits para comenzar a samplear
    last_id = x[0, -1].item()
    out_ids = ids.copy()

    for _ in range(max_new_tokens):
        # alimentar el último token
        inp = torch.tensor([[last_id]], dtype=torch.long, device=device)
        logits, h = model(inp, h0=h)      # (1,1,V)
        logits = logits[0, -1, :]         # (V,)

        # temperatura
        logits = logits / max(1e-8, temperature)
        probs = torch.softmax(logits, dim=-1)

        # muestreo
        next_id = torch.multinomial(probs, num_samples=1).item()
        out_ids.append(next_id)
        last_id = next_id

    return decode(out_ids, itos)

In [7]:
# -----------------------------
# 6) Script principal
# -----------------------------
def main(text_path: str, cfg: TrainConfig):
    text = load_text(text_path)
    #usar un texto con acentos/ñ y normalizar si hay caracteres raros (o aceptar que el vocabulario crece)
    stoi, itos = build_vocab(text)
    data_ids = encode(text, stoi)

    # split simple train/val
    n = len(data_ids)
    split = int(0.9 * n)
    train_ids = data_ids[:split]
    val_ids = data_ids[split:]

    train_ds = CharWindowDataset(train_ids, seq_len=cfg.seq_len)
    val_ds = CharWindowDataset(val_ids, seq_len=cfg.seq_len)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

    device = torch.device(cfg.device)
    model = CharGRULM(
        vocab_size=len(stoi),
        emb_dim=cfg.emb_dim,
        hidden_dim=cfg.hidden_dim,
        num_layers=cfg.num_layers
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    print(f"Vocab size: {len(stoi)} | Train windows: {len(train_ds)} | Val windows: {len(val_ds)}")
    for epoch in range(1, cfg.epochs + 1):
        tr = train_one_epoch(model, train_loader, optimizer, device)
        vl = eval_loss(model, val_loader, device)
        ppl = math.exp(vl) if vl < 20 else float("inf")
        print(f"Epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")

    # demos de generación
    prompt = "La inteligencia artificial "
    for temp in [0.7, 1.0, 1.2]:
        sample = generate(model, prompt, stoi, itos, max_new_tokens=400, temperature=temp, device=device)
        print("\n" + "="*60)
        print(f"Temperature = {temp}")
        print(sample)


In [ ]:
if __name__ == "__main__":
    # Cambia la ruta al archivo de texto
    # Si el entrenamiento es lento:
    # -bajar hidden_dim a 128,
    # -bajar num_layers a 1,
    # -bajar seq_len a 64,
    # -subir batch_size solo si hay RAM.
    # -----------------------------
    cfg = TrainConfig(
        seq_len=128,
        batch_size=64,
        emb_dim=128,
        hidden_dim=256,
        num_layers=2,
        lr=3e-4,
        epochs=5,
        device="cpu"  # cambia a "cuda" si tienes GPU
    )
    main("texto.txt", cfg)